In [1]:
%load_ext autoreload
%autoreload 2

## 1. Imports

In [2]:
import os
import sys

sys.path.append("..")

import numpy as np
import torch
from tqdm import tqdm

import wandb
from src.distributions import StandardNormalSampler, SwissRollSampler, PairedSampler
from src.light_gcot import LightGCOT

In [3]:
device = torch.device(f"cuda:{torch.cuda.current_device()}" if torch.cuda.is_available() else "cpu")
device

device(type='cuda', index=0)

In [4]:
torch.set_default_device(device)
torch.torch.set_default_dtype(torch.float64)

## 2. Config

In [5]:
X_DIM = 2
Y_DIM = 2
assert X_DIM > 1
assert Y_DIM > 1

OUTPUT_SEED = 42

N_POTENTIALS = 75
M_POTENTIALS = 25
EPSILON = 0.002
INIT_BY_SAMPLES = False
A_DIAGONAL_INIT = 0.5

BATCH_SIZE = 128
SAMPLING_BATCH_SIZE = 128

D_LR = 3e-4  # 1e-3 for eps 0.1, 0.01 and 3e-4 for eps 0.002
D_GRADIENT_MAX_NORM = float("inf")

N_PAIRED_SAMPLES = 16
M_UNPAIRED_SAMPLES = 16384

PLOT_EVERY = 1000
MAX_STEPS = 50000
CONTINUE = -1

In [6]:
torch.manual_seed(OUTPUT_SEED)
np.random.seed(OUTPUT_SEED)

In [7]:
EXP_COST = "MLP"
EXP_COST_INCLUDED = True
EXP_META_INFO = ""
EXP_NAME = (
    f"LightGCOT_Swiss_Roll_EPSILON_{EPSILON}_MAX_STEPS_{MAX_STEPS}_N_{N_POTENTIALS}_M_{M_POTENTIALS}_with_{EXP_COST}_cost_included_{EXP_COST_INCLUDED}_N_PAIRED_{N_PAIRED_SAMPLES}_M_UNPAIRED_{M_UNPAIRED_SAMPLES}"
    + EXP_META_INFO
)
OUTPUT_PATH = "../checkpoints/{}".format(EXP_NAME)

config = dict(
    X_DIM=X_DIM,
    Y_DIM=Y_DIM,
    D_LR=D_LR,
    BATCH_SIZE=BATCH_SIZE,
    EPSILON=EPSILON,
    D_GRADIENT_MAX_NORM=D_GRADIENT_MAX_NORM,
    N_POTENTIALS=N_POTENTIALS,
    M_POTENTIALS=M_POTENTIALS,
    INIT_BY_SAMPLES=INIT_BY_SAMPLES,
    A_DIAGONAL_INIT=A_DIAGONAL_INIT,
    N_PAIRED_SAMPLES=N_PAIRED_SAMPLES,
    M_UNPAIRED_SAMPLES=M_UNPAIRED_SAMPLES,
)

if not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH)

## 3. Create samplers

In [8]:
X_sampler = StandardNormalSampler(dim=2, device="cuda")
Y_sampler = SwissRollSampler(dim=2, device="cuda")

## 4. Model initialization

In [9]:
D = LightGCOT(
    x_dim=X_DIM,
    y_dim=Y_DIM,
    n_potentials=N_POTENTIALS,
    m_potentials=M_POTENTIALS,
    epsilon=EPSILON,
    sampling_batch_size=SAMPLING_BATCH_SIZE,
    A_diagonal_init=A_DIAGONAL_INIT,
    cost_function=EXP_COST,
)

if INIT_BY_SAMPLES:
    D.init_a_by_samples(Y_sampler.sample(N_POTENTIALS))

D_opt = torch.optim.Adam(D.parameters(), lr=D_LR)

if CONTINUE > -1:
    D_opt.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f"D_opt_{OUTPUT_SEED}_{CONTINUE}.pt")))

## 5. Model training

In [10]:
from src.discrete_ot import OTPlanSampler
from src.plotting import plot_A_parameters, plot_B_parameters, plot_distributions

In [11]:
otp_sampler = OTPlanSampler("sinkhorn")

In [13]:
sampler = PairedSampler(X_sampler, Y_sampler, BATCH_SIZE, N_PAIRED_SAMPLES, M_UNPAIRED_SAMPLES, 128, otp_sampler)

In [14]:
starting_points = torch.tensor([[-1.5, 1.5], [0.0, 0.0], [1.5, -1.5]])

In [15]:
wandb.init(name=EXP_NAME, config=config)

for step in tqdm(range(CONTINUE + 1, MAX_STEPS)):
    # training loop
    D_opt.zero_grad()

    X, Y = sampler.sample()
    # X, Y = sampler.sample_pair()

    log_v_m = D.compute_log_v_m(X)  # [bs x M]
    b_m = D.compute_b_m(X)  # [bs x M x y_dim]

    log_w_n = D.compute_log_w_n()  # [N]
    a_n = D.compute_a_n()  # [N x y_dim]
    A_n = D.compute_A_n()  # [N x y_dim]

    f_c = D.compute_dual_potential(log_w_n, a_n, A_n, log_v_m, b_m)
    f = D.compute_primal_potential(Y, log_w_n, a_n, A_n)

    if EXP_COST_INCLUDED:
        X_paired, Y_paired = sampler.sample_pair()
        log_v_m_paired = D.compute_log_v_m(X_paired)  # [bs x M]
        b_m_paired = D.compute_b_m(X_paired)  # [bs x M x y_dim]

        c = D.compute_cost(Y_paired, log_v_m_paired, b_m_paired)
        # c = D.compute_cost(Y, log_v_m, b_m)
        # X_paired, Y_paired = X, Y
        D_loss = c.mean() - (f_c + f).mean()
        D_loss.backward()
        wandb.log({r"$c(x, y)$": c.mean().item()}, step=step)
    else:
        D_loss = -(f_c + f).mean()
        D_loss.backward()
    D_gradient_norm = torch.nn.utils.clip_grad_norm_(D.parameters(), max_norm=D_GRADIENT_MAX_NORM)
    D_opt.step()

    wandb.log({f"D gradient norm": D_gradient_norm.item()}, step=step)
    wandb.log({f"D_loss": D_loss.item()}, step=step)
    wandb.log({r"$-f^c(x)$": -f_c.mean().item()}, step=step)
    wandb.log({r"$-f(y)$": -f.mean().item()}, step=step)
    wandb.log({r"$-f(y)-f^c(x)$": -(f_c + f).mean().item()}, step=step)
    wandb.log({f"lam_min(A_n)": torch.min(A_n)}, step=step)
    wandb.log({f"lam_max(A_n)": torch.max(A_n)}, step=step)

    if step % PLOT_EVERY == 0:
        A_dict = plot_A_parameters(D, log=True)
        B_dict = plot_B_parameters(D, starting_points, log=True)
        distr_dict = plot_distributions(D, X_sampler, Y_sampler, X_paired, Y_paired, starting_points, log=True)
        wandb.log(A_dict | B_dict | distr_dict)

        torch.save(D.state_dict(), os.path.join(OUTPUT_PATH, f"D_{step}.pt"))
        torch.save(D_opt.state_dict(), os.path.join(OUTPUT_PATH, f"D_opt_{step}.pt"))

torch.save(D.state_dict(), os.path.join(OUTPUT_PATH, f"D_{MAX_STEPS}.pt"))
torch.save(D_opt.state_dict(), os.path.join(OUTPUT_PATH, f"D_opt_{MAX_STEPS}.pt"))

wandb.finish()

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: muxaujl11110. Use `wandb login --relogin` to force relogin


  0%|                                                                                                                                                                                 | 0/50000 [00:00<?, ?it/s]/beegfs/home/m.persiyanov/Light-GCOT/src/plotting.py:235: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(loc="lower right")
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50000/50000 [14:53<00:00, 55.96it/s]


$-f(y)$,▃▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▃▂▃▃▃▃▄▅▄▅▅▅▅▅▆▆█▆▇▆███
$-f(y)-f^c(x)$,█▅▄▃▃▂▂▂▂▁▁▁▁▁▂▂▂▂▂▂▂▂▂▂▃▂▃▃▃▂▂▃▂▅▁▄▂▃▃▃
$-f^c(x)$,█▇▇▆▆▆▆▅▅▅▅▅▅▅▅▅▅▄▄▄▄▄▄▄▄▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁
"$c(x, y)$",████████████▇▇▇▇▆▆▆▅▅▅▅▄▄▄▄▄▃▃▃▃▃▂▂▂▂▁▁▁
D gradient norm,█▃▂▂▂▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▃▃▃▄▅▄▅▄▄▄▅▇
D_loss,█▇▇▆▆▆▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▃▁▂▁▁▁▁
lam_max(A_n),▁▁▁▁▂▂▂▃▃▄▅▆▇████▇██▇▇▇▆▆▆▆▆▅▅▅▅▅▅▅▅▄▄▄▄
lam_min(A_n),█▇▆▅▄▄▄▄▄▄▃▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
$-f(y)$,6.45269
$-f(y)-f^c(x)$,-0.8677
$-f^c(x)$,-7.32039


## Plotting

In [ ]:
plot_distributions(D, X_sampler, Y_sampler, starting_points)